# 02 — Structural sparsity and visualization

**Objectives.** Distinguish structural and numerical zeros, inspect CCS metadata, and visualize Jacobian patterns.

CasADi stores sparse matrices in compressed-column storage (CCS). A *structural nonzero* reserves a graph entry even if its current numerical value is zero. `nnz()` counts structural entries; `numel()` counts every matrix location.


In [ ]:
import casadi as ca
import matplotlib.pyplot as plt
from casadi_tutorial.plotting import plot_sparsity
sp = ca.Sparsity.triplet(5, 5, [0, 1, 2, 3, 4, 0], [0, 1, 2, 3, 4, 4])
A = ca.DM(sp, [1, 0, 3, 4, 5, 6])
print('shape', A.shape, 'nnz', A.nnz(), 'numel', A.numel())
print('CCS column offsets:', A.sparsity().colind())
print('stored rows:', A.sparsity().row())
plot_sparsity(A, title='Structural nonzeros (one stored value is zero)')


In [ ]:
x = ca.SX.sym('x', 6)
f = ca.vertcat(x[0]**2 + x[2], ca.sin(x[1]) + x[4]*x[5], x[3])
J = ca.jacobian(f, x)
fig, ax = plt.subplots(figsize=(6, 3))
plot_sparsity(J, ax=ax, title='Jacobian pattern')
assert J.nnz() == 6


## Why sparsity matters

A multiple-shooting Jacobian is block-banded: each continuity equation touches only neighboring states and one control. Sparse factorization exploits this locality. Accidentally densifying a graph increases derivative storage and KKT factorization cost.

**Common mistake:** `ca.DM.zeros(n,n)` is structurally dense; construct a `Sparsity` pattern when the structure matters. **Recap:** plot the Jacobian and Hessian structure before tuning a large solver.
